# ⚡ 06 · How an electron microscope makes an image
### Projection, dose and the contrast transfer function (CTF) — simulated with a β-galactosidase-sized map

**BiologyToResearch** · Part B — Imaging physics · Module 06 of 34

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearch/notebooks/06_electron_image_formation.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy, Matplotlib |
| **Prerequisites** | Module 03 (Fourier transforms, frequencies in Å⁻¹, filters) and Module 04 (noise, SNR, averaging) |
| **Estimated time** | 120 min (reading + running + the ✏️ exercises) |
| **Lab papers covered** | Microscope settings of the Xu-lab simulators (Xu, Beck & Alber 2011, 2012; Pei *et al.* 2016; the 2017+ deep-learning benchmark; SHREC'20); context for 2D cryo-EM papers: CryoMAE (2025), score-based denoising with target guidance (2026), simulated-annealing image selection (Shi *et al.* 2020) — see [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> All experiments in this course are small, controlled *toy versions* of published methods. They are designed to teach
> the core idea of each paper; they do **not** reproduce the papers' numbers. **The density map is generated by code**
> (nothing is downloaded): a β-galactosidase-sized tetramer built from ideal α-helices at 3.54 Å per pixel — a
> realistic stand-in, not the real enzyme's structure. Image formation is simplified (weak-phase, projection
> approximation, no inelastic scattering, no detector model beyond counting noise).

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** why samples are vitrified and imaged with few electrons, and why a cryo-EM image is approximately a
   projection of the specimen's Coulomb potential.
2. **Compute** the relativistic electron wavelength (0.0197 Å at 300 kV) and **write down** the CTF,
   $\chi(k)=\pi\lambda\Delta f k^2-\tfrac{\pi}{2}C_s\lambda^3k^4$, stating the sign convention.
3. **Simulate** a micrograph (projections → CTF → electron counting noise) and **see** Thon rings in its power spectrum.
4. **Estimate** the defocus of a micrograph by fitting the CTF to its Thon rings.
5. **Measure** how dose controls the SNR, and **explain** why radiation damage caps the dose.
6. **Correct** the CTF by phase flipping and Wiener filtering, and **explain** why data are collected at several defoci.
7. **Relate** pixel size, Nyquist frequency and binning to what a tomogram can resolve, and **compare** single-particle
   cryo-EM with cryo-ET.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit, the β-galactosidase map | 5 min |
| 1 · Freezing, and why an image is a projection | the weak-phase / projection approximation | 10 min |
| 2 · Particles in random orientations | project the map like particles in ice | 8 min |
| 3 · The electron wavelength | relativistic λ and the interaction constant | 8 min |
| 4 · The contrast transfer function | χ(k), defocus, C_s, amplitude contrast, envelopes | 20 min |
| 5 · Thon rings and defocus estimation | a simulated micrograph and its power spectrum | 15 min |
| 6 · Dose, shot noise and radiation damage | Poisson counting; SNR ∝ dose; critical exposure | 15 min |
| 7 · Correcting the CTF | phase flipping and Wiener filtering | 12 min |
| 8 · Pixel size, Nyquist and binning | what a 10 Å voxel can see | 8 min |
| 9 · Single-particle vs tomography; 2D class averages | CTF-corrected averaging over many defoci | 12 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 10 min |

## 🧫 The biological question

Biological molecules are made of light atoms (H, C, N, O) bathed in water. In an electron microscope they barely change
the *amplitude* of the electron beam — an in-focus image of a protein in ice is almost perfectly grey. Worse, electrons
break chemical bonds: after a few tens of electrons per square ångström, the fine structure is gone. And the vacuum of the
microscope would boil away liquid water.

Three tricks make cryo-EM possible: **freeze** the sample so fast that water becomes glass; **defocus** the microscope so
that invisible phase shifts turn into visible contrast — at the price of a characteristic "ringing" described by the
**contrast transfer function (CTF)**; and **use so few electrons** that every image is buried in counting noise, then
recover the signal computationally. Every algorithm in the rest of this course works on images formed this way, so we now
build that image formation step by step with a high-resolution map shaped like the enzyme **β-galactosidase**.

---
## 0 · Environment setup

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, this notebook only needs a CPU.")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch not installed (not needed in this notebook).")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import os
import time

import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage

SEED = 0
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

Copied verbatim from [`course_toolkit.py`](../course_toolkit.py): map building, rotations,
Fourier helpers, the electron wavelength and the 2D CTF (`ctf_2d`) that every later simulation in the course uses.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import struct

import numpy as np
from scipy import ndimage

_BOND = dict(N_CA=1.458, CA_C=1.525, C_N=1.329, C_O=1.231)
_ANGLE = dict(N_CA_C=111.2, CA_C_N=116.2, C_N_CA=121.7, CA_C_O=120.5)
_SIDECHAIN = {
    "GLY": [], "ALA": [], "SER": [("OG", "O")], "CYS": [("SG", "S")], "VAL": [("CG1", "C"), ("CG2", "C")],
    "THR": [("OG1", "O"), ("CG2", "C")], "PRO": [("CG", "C"), ("CD", "C")],
    "LEU": [("CG", "C"), ("CD1", "C"), ("CD2", "C")], "ILE": [("CG1", "C"), ("CG2", "C"), ("CD1", "C")],
    "ASP": [("CG", "C"), ("OD1", "O"), ("OD2", "O")], "ASN": [("CG", "C"), ("OD1", "O"), ("ND2", "N")],
    "GLU": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("OE2", "O")], "GLN": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("NE2", "N")],
    "LYS": [("CG", "C"), ("CD", "C"), ("CE", "C"), ("NZ", "N")], "MET": [("CG", "C"), ("SD", "S"), ("CE", "C")],
    "ARG": [("CG", "C"), ("CD", "C"), ("NE", "N"), ("CZ", "C"), ("NH1", "N"), ("NH2", "N")],
    "HIS": [("CG", "C"), ("ND1", "N"), ("CD2", "C"), ("CE1", "C"), ("NE2", "N")],
    "PHE": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C")],
    "TYR": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C"), ("OH", "O")],
    "TRP": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("NE1", "N"), ("CE2", "C"), ("CE3", "C"), ("CZ2", "C"),
            ("CZ3", "C"), ("CH2", "C")],
}
_AA_FREQ = {"LEU": 9.9, "ALA": 8.3, "GLY": 7.1, "VAL": 6.9, "GLU": 6.8, "SER": 6.6, "ILE": 5.9, "LYS": 5.8, "ARG": 5.5,
            "ASP": 5.5, "THR": 5.3, "PRO": 4.7, "ASN": 4.1, "GLN": 3.9, "PHE": 3.9, "TYR": 2.9, "MET": 2.4, "HIS": 2.3,
            "CYS": 1.4, "TRP": 1.1}
def _nerf(a, b, c, bond, angle, torsion):
    """Place atom d so that |cd| = bond, angle(b,c,d) = angle and dihedral(a,b,c,d) = torsion (degrees)."""
    bc = (c - b) / np.linalg.norm(c - b)
    n = np.cross(b - a, bc)
    n /= np.linalg.norm(n)
    m = np.cross(n, bc)
    ang, tor = np.radians(angle), np.radians(torsion)
    d = np.array([-bond * np.cos(ang), bond * np.sin(ang) * np.cos(tor), bond * np.sin(ang) * np.sin(tor)])
    return c + d[0] * bc + d[1] * m + d[2] * n


def build_backbone(phi, psi, omega=None) -> np.ndarray:
    """Backbone atoms N, CA, C, O, CB (array n×5×3, Å) of a chain with the given torsion angles (degrees)."""
    n = len(phi)
    omega = np.full(n, 180.0) if omega is None else np.asarray(omega, float)
    N = [np.zeros(3)]
    CA = [np.array([_BOND["N_CA"], 0.0, 0.0])]
    C = [_nerf(np.array([0.0, 1.0, 0.0]), N[0], CA[0], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[0])]
    for i in range(n - 1):
        N.append(_nerf(N[i], CA[i], C[i], _BOND["C_N"], _ANGLE["CA_C_N"], psi[i]))
        CA.append(_nerf(CA[i], C[i], N[i + 1], _BOND["N_CA"], _ANGLE["C_N_CA"], omega[i]))
        C.append(_nerf(C[i], N[i + 1], CA[i + 1], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[i + 1]))
    N, CA, C = np.array(N), np.array(CA), np.array(C)
    O = np.array([_nerf(N[i], CA[i], C[i], _BOND["C_O"], _ANGLE["CA_C_O"], psi[i] + 180.0) for i in range(n)])
    b, c = CA - N, C - CA
    a = np.cross(b, c)
    CB = -0.58273431 * a + 0.56802827 * b - 0.54067466 * c + CA      # ideal Cβ position
    return np.stack([N, CA, C, O, CB], 1)


def _ideal_helix_template(n=160):
    """An ideal α-helix (φ = −57.8°, ψ = −47.0°) with its axis along +z through x = y = 0."""
    bb = build_backbone(np.full(n, -57.8), np.full(n, -47.0))
    ca = bb[:, 1]
    centre = ca.mean(0)
    u = np.linalg.svd(ca - centre)[2][0]
    if np.dot(ca[-1] - ca[0], u) < 0:
        u = -u
    # rotation taking u to z
    v = np.cross(u, [0, 0, 1.0]); s = np.linalg.norm(v); cth = u[2]
    K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    Rm = np.eye(3) + K + K @ K * ((1 - cth) / (s ** 2 + 1e-12))
    xyz = (bb - centre) @ Rm.T
    axis_xy = xyz[:, 1, :2].mean(0)
    xyz[..., :2] -= axis_xy
    return xyz


_HELIX = _ideal_helix_template()
def _sidechains(bb, resnames, rng):
    """Approximate side-chain heavy atoms grown outward from Cβ (positions are schematic; counts/elements are right)."""
    names, elems, xyz, resid = [], [], [], []
    ca, cb = bb[:, 1], bb[:, 4]
    out = cb - ca
    out /= np.linalg.norm(out, axis=1, keepdims=True)
    for i, rn in enumerate(resnames):
        sc = _SIDECHAIN[rn]
        if not sc:
            continue
        perp = np.cross(out[i], rng.normal(size=3)); perp /= np.linalg.norm(perp)
        for k, (an, el) in enumerate(sc):
            depth = 1.0 + (k // 2)                               # schematic zig-zag outward
            pos = cb[i] + 1.5 * depth * out[i] + 0.9 * (1 if k % 2 else -1) * perp
            names.append(an); elems.append(el); xyz.append(pos); resid.append(i)
    return names, elems, np.array(xyz).reshape(-1, 3), np.array(resid, int)


def _loop_path(a, b, outward, step=3.8):
    """Cα positions of a loop from Cα `a` to Cα `b`: a circular arc bulging along `outward`, cut into equal chords so
    that every Cα–Cα distance (including the two ends) is exactly `step` Å."""
    c = np.linalg.norm(b - a)
    e = (b - a) / c
    u = outward - (outward @ e) * e                                  # bulge direction ⟂ to the chord
    if np.linalg.norm(u) < 1e-6:
        u = np.cross(e, [0.0, 0.0, 1.0]) if abs(e[2]) < 0.9 else np.cross(e, [1.0, 0.0, 0.0])
    u = u / np.linalg.norm(u)
    m = max(int(np.ceil(c / step)), 1)                               # m new residues → n = m + 1 equal chords
    n = m + 1
    lo, hi = 1e-9, 2 * np.pi - 1e-9                                  # arc angle Θ: sin(Θ/2) / sin(Θ/2n) = c / step
    for _ in range(60):
        th = (lo + hi) / 2
        lo, hi = (th, hi) if np.sin(th / 2) / np.sin(th / (2 * n)) > c / step else (lo, th)
    R = step / (2 * np.sin(th / (2 * n)))
    centre = (a + b) / 2 - R * np.cos(th / 2) * u
    beta = np.pi / 2 + th / 2 - th * np.arange(1, n) / n             # angles of the new points, a → b over the bulge
    return centre + R * (np.outer(np.cos(beta), e) + np.outer(np.sin(beta), u))


def _assemble_chain(ca_path_segments, rng, chain="A", start_resnum=1):
    """Turn helices (n×5×3 arrays) joined by loops into one chain with residue names and side chains."""
    blocks = []
    for k, seg in enumerate(ca_path_segments):
        blocks.append(seg)
        if k + 1 < len(ca_path_segments):
            a, b = seg[-1, 1], ca_path_segments[k + 1][0, 1]
            path = _loop_path(a, b, (a + b) / 2 - seg.reshape(-1, 3).mean(0))
            m = len(path)
            tang = np.gradient(path, axis=0) if m > 1 else np.array([b - a])
            tang /= np.linalg.norm(tang, axis=1, keepdims=True)
            side = np.cross(tang, rng.normal(size=3)); side /= np.linalg.norm(side, axis=1, keepdims=True)
            loop = np.stack([path - 1.2 * tang, path, path + 1.2 * tang, path + 1.2 * tang + 1.0 * side,
                             path - 1.3 * side], 1)
            blocks.append(loop)
    bb = np.concatenate(blocks, 0)
    aa = list(_AA_FREQ)
    p = np.array([_AA_FREQ[a] for a in aa]); p /= p.sum()
    resnames = rng.choice(aa, len(bb), p=p)
    bb_names = ["N", "CA", "C", "O", "CB"]
    names, elems, xyz, resid = [], [], [], []
    for i in range(len(bb)):
        for j, an in enumerate(bb_names):
            if an == "CB" and resnames[i] == "GLY":
                continue
            names.append(an); elems.append(an[0]); xyz.append(bb[i, j]); resid.append(i)
    sn, se, sx, sr = _sidechains(bb, resnames, rng)
    names += sn; elems += se; resid = np.concatenate([np.array(resid, int), sr])
    xyz = np.concatenate([np.array(xyz), sx], 0)
    order = np.argsort(resid, kind="stable")
    return dict(xyz=xyz[order].astype(np.float32), name=np.array(names)[order], element=np.array(elems)[order],
                resname=resnames[resid[order]], resnum=(resid[order] + start_resnum).astype(int),
                chain=np.full(len(order), chain))


def helix_bundle(radii_A, rng, chain="A", spacing=10.0):
    """A globular 'domain' that fills an ellipsoid (semi-axes in Å, long axis = local z) with antiparallel helices."""
    a, b, c = radii_A[0] + 2.5, radii_A[1] + 2.5, radii_A[2] + 1.5   # helices stop short of the surface: pad
    rows = []
    j = 0
    while (j * spacing * np.sqrt(3) / 2) <= b:
        rows.append(j); rows.append(-j) if j else None; j += 1
    pts = []
    for jj in sorted(set(rows)):
        y = jj * spacing * np.sqrt(3) / 2
        xs = np.arange(-a, a + 1e-6, spacing) + (spacing / 2 if jj % 2 else 0)
        row = [(x, y) for x in xs if (x / a) ** 2 + (y / b) ** 2 < 1]
        pts.append(row if (jj % 2 == 0) else row[::-1])
    pts = [p for row in pts for p in row]
    segs = []
    for k, (x, y) in enumerate(pts):
        h = c * np.sqrt(max(0.0, 1 - (x / a) ** 2 - (y / b) ** 2)) - 2.0
        n = int(2 * h / 1.5)
        if n < 6:
            continue
        seg = _HELIX[:n].copy()
        seg[..., 2] -= seg[:, 1, 2].mean()
        phase = rng.uniform(0, 2 * np.pi)
        Rz = np.array([[np.cos(phase), -np.sin(phase), 0], [np.sin(phase), np.cos(phase), 0], [0, 0, 1]])
        seg = seg @ Rz.T
        if len(segs) % 2 == 1:                                   # antiparallel neighbours: 180° turn about x
            seg = seg * np.array([1.0, -1.0, -1.0])
        seg = seg + np.array([x, y, rng.uniform(-1.0, 1.0)])
        segs.append(seg)
    return _assemble_chain(segs, rng, chain)


def _transform(atoms, R=np.eye(3), t=(0, 0, 0)):
    out = dict(atoms)
    out["xyz"] = (atoms["xyz"] @ np.asarray(R, float).T + np.asarray(t, float)).astype(np.float32)
    return out


def merge_atoms(parts):
    keys = parts[0].keys()
    return {k: np.concatenate([p[k] for p in parts]) for k in keys}


def _rz(a):
    return np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1.0]])


def _rx(a):
    return np.array([[1, 0, 0], [0, np.cos(a), -np.sin(a)], [0, np.sin(a), np.cos(a)]])


def _ry(a):
    return np.array([[np.cos(a), 0, np.sin(a)], [0, 1, 0], [-np.sin(a), 0, np.cos(a)]])


_Z = {"C": 6.0, "N": 7.0, "O": 8.0, "S": 16.0, "P": 15.0, "H": 1.0}
def atoms_to_map(xyz, weights=None, voxel=10.0, box=None, sigma_A=None):
    """Density map [z, y, x] from atom positions (Å): trilinear splatting on a grid twice as fine, Gaussian blur
    (σ = `sigma_A`, default = voxel/2), then 2×2×2 binning to `voxel` Å. Centred on the box if `box` is given."""
    xyz = np.asarray(xyz, float)
    w = np.ones(len(xyz)) if weights is None else np.asarray(weights, float)
    fine = voxel / 2.0
    if box is None:
        ext = np.ptp(xyz, 0).max() + 6 * voxel
        box = int(np.ceil(ext / voxel)) + 1
    n = 2 * box
    centre = (n - 1) / 2.0
    p = xyz[:, ::-1] / fine + centre                                  # (z, y, x) grid coordinates
    grid = np.zeros((n, n, n), np.float32)
    i0 = np.floor(p).astype(int)
    f = p - i0
    for dz in (0, 1):
        for dy in (0, 1):
            for dx in (0, 1):
                idx = i0 + np.array([dz, dy, dx])
                wt = w * np.where(dz, f[:, 0], 1 - f[:, 0]) * np.where(dy, f[:, 1], 1 - f[:, 1]) * np.where(dx, f[:, 2], 1 - f[:, 2])
                ok = np.all((idx >= 0) & (idx < n), 1)
                np.add.at(grid, tuple(idx[ok].T), wt[ok])
    grid = ndimage.gaussian_filter(grid, (sigma_A if sigma_A else voxel / 2.0) / fine)
    return grid.reshape(box, 2, box, 2, box, 2).mean((1, 3, 5)).astype(np.float32)


def betagal_like_map(voxel=3.54, box=100, b_factor=40.0):
    """A β-galactosidase-sized D2 homotetramer (code-generated, ~4 × 1,000 residues) at `voxel` Å, `box`³ voxels.

    Returns (map[z, y, x], voxel, source, atoms). Atomic B-factor `b_factor` (Å²) blurs each atom by
    σ = √(B / 8π²); helices (≈ 10 Å apart) are visible at this sampling, side chains are not.
    """
    rng = np.random.default_rng(1234)
    domains = [((21.0, 18.0, 22.0), (0.0, 0.0, 0.0)), ((18.0, 16.0, 20.0), (34.0, 8.0, 4.0)),
               ((17.0, 15.0, 19.0), (-31.0, 10.0, -3.0)), ((16.0, 15.0, 18.0), (6.0, 31.0, 8.0)),
               ((15.0, 14.0, 17.0), (-8.0, -29.0, 5.0))]
    parts = [_transform(helix_bundle(r, rng), _rz(rng.uniform(0, 6.3)), c) for r, c in domains]
    mono = merge_atoms(parts)
    mono["xyz"] = mono["xyz"] - mono["xyz"].mean(0) + np.array([30.0, 21.0, 13.0])
    subunits = []
    for k, R in enumerate([np.eye(3), _rz(np.pi), _rx(np.pi), _ry(np.pi)]):          # D2: identity + three 2-folds
        s = _transform(mono, R)
        s["chain"] = np.full(len(s["xyz"]), "ABCD"[k])
        subunits.append(s)
    atoms = merge_atoms(subunits)
    atoms["xyz"] -= atoms["xyz"].mean(0)
    w = np.array([_Z.get(e, 6.0) for e in atoms["element"]])
    sigma = np.sqrt(b_factor / (8 * np.pi ** 2) + (voxel / 2.0) ** 2)                  # B-factor + sampling blur
    vol = atoms_to_map(atoms["xyz"], w, voxel=voxel, box=box, sigma_A=sigma)
    return vol, voxel, "code-generated β-galactosidase-sized D2 tetramer", atoms


def random_rotation(rng: np.random.Generator) -> np.ndarray:
    """Uniformly random proper rotation (det = +1) from a random unit quaternion."""
    q = rng.normal(size=4)
    q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


_P = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]], float)  # (x,y,z) <-> (z,y,x)
def rotate_volume(vol: np.ndarray, R: np.ndarray, shift_xyz=(0, 0, 0), order: int = 1,
                  cval: float = 0.0) -> np.ndarray:
    """Rotate a volume about its centre by R (then translate by shift, in voxels, (x,y,z)).

    out(p) = vol(Rᵀ (p − c − t) + c): the *object* is rotated by R and moved by t.
    """
    R_idx = _P @ R @ _P                       # same rotation written in (z, y, x) index order
    c = (np.array(vol.shape) - 1) / 2.0
    t_idx = _P @ np.asarray(shift_xyz, float)
    M = R_idx.T                               # inverse rotation maps output -> input
    offset = c - M @ (c + t_idx)
    return ndimage.affine_transform(vol, M, offset=offset, order=order, mode="constant", cval=cval)


def freq_grid(shape):
    """Frequency coordinates (cycles/voxel) for an unshifted FFT; returns list per axis."""
    return np.meshgrid(*[np.fft.fftfreq(n) for n in shape], indexing="ij")


def radial_frequency(shape) -> np.ndarray:
    """|k| in cycles/voxel for every coefficient of an unshifted FFT."""
    return np.sqrt(sum(k**2 for k in freq_grid(shape)))


def apply_fourier_mask(vol: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Multiply the FFT of ``vol`` by ``mask`` (unshifted layout) and return the real part."""
    return np.real(np.fft.ifftn(np.fft.fftn(vol) * mask)).astype(np.float32)


def add_noise_snr(signal: np.ndarray, snr: float, rng: np.random.Generator) -> np.ndarray:
    """Add white Gaussian noise so that var(signal)/var(noise) = snr."""
    sigma = np.sqrt(signal.var() / snr)
    return (signal + rng.normal(0, sigma, signal.shape)).astype(np.float32)


def pearson(a: np.ndarray, b: np.ndarray) -> float:
    """Pearson correlation of two arrays (flattened)."""
    a = a.ravel() - a.mean()
    b = b.ravel() - b.mean()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


def electron_wavelength_A(kV: float = 300.0) -> float:
    """Relativistic electron wavelength in Å (0.0197 Å at 300 kV)."""
    V = kV * 1e3
    return 12.2643 / np.sqrt(V * (1 + 0.978466e-6 * V))


def ctf_2d(shape, pixel_A: float, defocus_um: float = 3.0, kV: float = 300.0,
           Cs_mm: float = 2.7, amp_contrast: float = 0.07, B: float = 0.0) -> np.ndarray:
    """Contrast transfer function on an unshifted FFT grid (underfocus is positive).

    CTF(k) = −[√(1−A²)·sin χ(k) + A·cos χ(k)]·exp(−B k²/4),
    χ(k) = π λ Δf k² − (π/2) C_s λ³ k⁴, with k in 1/Å.
    """
    lam = electron_wavelength_A(kV)
    k2 = sum(kk**2 for kk in freq_grid(shape)) / pixel_A**2
    dz = defocus_um * 1e4
    cs = Cs_mm * 1e7
    chi = np.pi * lam * dz * k2 - 0.5 * np.pi * cs * lam**3 * k2**2
    ctf = -(np.sqrt(1 - amp_contrast**2) * np.sin(chi) + amp_contrast * np.cos(chi))
    return (ctf * np.exp(-B * k2 / 4)).astype(np.float32)


def show_orthoslices(vol, title: str = "", axes=None, cmap: str = "gray"):
    """Show the three central orthogonal slices (xy, xz, yz) of a volume."""
    import matplotlib.pyplot as plt
    if axes is None:
        _, axes = plt.subplots(1, 3, figsize=(7.5, 2.6))
    z, y, x = (np.array(vol.shape) // 2)
    for ax, im, lab in zip(axes, [vol[z], vol[:, y, :], vol[:, :, x]], ["xy (z mid)", "xz (y mid)", "yz (x mid)"]):
        ax.imshow(im, cmap=cmap)
        ax.set_title(f"{title} {lab}".strip(), fontsize=8)
        ax.axis("off")
    return axes
print("🧰 toolkit loaded")

### Data: a high-resolution map of a β-galactosidase-sized tetramer

🧫 **β-galactosidase** (*E. coli* LacZ) is the enzyme that splits lactose — the "lac" of the lac operon. It is a
tetramer of four identical ≈ 116 kDa chains (≈ 465 kDa in total) with D2 symmetry, and a favourite test specimen of
cryo-EM. The toolkit builds a stand-in of the same size and symmetry — four identical subunits of five helical domains
each (≈ 4 × 1,100 residues), arranged with D2 symmetry — and samples it on a 100³ grid at **3.54 Å per pixel** (a 354 Å
cube), like a typical single-particle map. Real maps contain small negative values and background ripples (from
filtering), so we always **clip** negatives and apply a soft **mask** around the molecule before projecting it; on our
clean code-generated map these two steps change very little.

In [ ]:
# ── Build the β-galactosidase-sized map (Module 01, Section 4) ──
raw_map, PIX, MAP_SOURCE, _ = betagal_like_map()


def prepare_particle(vol: np.ndarray) -> np.ndarray:
    """Clip negative densities and multiply by a soft mask around the molecule (clean solvent background)."""
    smooth = ndimage.gaussian_filter(vol, 2.0)
    mask = ndimage.binary_dilation(smooth > 0.1 * smooth.max(), iterations=3)
    return (np.clip(vol, 0, None) * ndimage.gaussian_filter(mask.astype(np.float32), 2.0)).astype(np.float32)


betagal = prepare_particle(raw_map)
print(f"map: {raw_map.shape} voxels at {PIX} Å/voxel = {raw_map.shape[0] * PIX:.0f} Å box  ← {MAP_SOURCE}")
print(f"raw values {raw_map.min():+.3f} … {raw_map.max():+.3f}; molecule occupies {100 * (betagal > 0.02 * betagal.max()).mean():.1f} % of the box")
inside = np.argwhere(betagal > 0.2 * betagal.max())
print("extent of the molecule (z, y, x) ≈", np.round(np.ptp(inside, axis=0) * PIX).astype(int), "Å")

---
## 1 · Freezing, and why an image is a projection

> 📍 **Where we are.** Module 03 gave you Fourier transforms and filters; Module 04 gave you noise, SNR and averaging.
> This module is the physics that *creates* the noisy, filtered images: the projection, the CTF filter and the counting
> noise. Module 07 then combines many tilted images into a tomogram, and Module 09 simulates a whole crowded cell.

🧫 **Vitrification.** A thin film of sample (≈ 20–100 nm for single particles; up to a few hundred nm for cells or
FIB-milled lamellae) is plunged into liquid ethane at ≈ −180 °C. Cooling is so fast that water molecules have no time to
form ice crystals: the water becomes **vitreous ice**, a glass that keeps molecules in a near-native, hydrated state and
survives the microscope's vacuum.

🧮 **What the electrons "see".** A fast electron passing through the specimen is not absorbed; it is slightly
**delayed**, because it travels through the electrostatic (Coulomb) potential $V(x,y,z)$ of the atoms. For a thin
specimen, the wave leaving the specimen carries a phase shift proportional to the potential summed along the beam
direction $z$:

$$\psi(x,y) = e^{\,i\sigma V_p(x,y)} \approx 1 + i\,\sigma V_p(x,y), \qquad V_p(x,y) = \int V(x,y,z)\,dz .$$

*In words:* the specimen acts as a **phase object** — it shifts the electron wave without dimming it. $V_p$ is the
**projected potential** (the "sum along an axis" of Module 01), and $\sigma$ is the **interaction constant** (Section 3).
The approximation $e^{i\phi}\approx1+i\phi$ (**weak-phase approximation**) holds because biological phase shifts are small;
summing straight through the specimen (**projection approximation**) holds for thin samples. The problem: the recorded
intensity is $|\psi|^2 = 1 + \sigma^2V_p^2 \approx 1$ — **an in-focus image of a weak phase object has almost no
contrast.** Section 4 shows how defocus fixes this.

In [ ]:
# ── 1.1 Look at the map: central slices and projections ──
fig, axes = plt.subplots(2, 3, figsize=(8.5, 5.6))
show_orthoslices(betagal, title="slice", axes=axes[0])
for ax, axis, lab in zip(axes[1], [0, 1, 2], ["along z (beam)", "along y", "along x"]):
    ax.imshow(betagal.sum(axis=axis)); ax.axis("off"); ax.set_title(f"projection {lab}", fontsize=8)
axes[1, 0].add_patch(plt.Rectangle((5, 92), 100 / PIX, 3, color="w"))      # 100 Å scale bar
axes[1, 0].text(5, 88, "100 Å", color="w", fontsize=7)
fig.suptitle(f"β-galactosidase ({PIX} Å/px): slices (top) and projections V_p (bottom)", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Slices cut through the molecule and show internal detail; projections add up everything along
the beam, so features at different depths overlap. A microscope records only projections. The bright spots in the
projections are places where much protein lies along the beam. The scale bar is 100 Å = 10 nm.

---
## 2 · Particles in random orientations

In single-particle cryo-EM, thousands of copies of the purified molecule are trapped in the ice in **random
orientations**, so each particle image is a projection from a different direction. (In cryo-ET we instead tilt the same
specimen — Module 07.) We rotate the 3D map by uniformly random rotations (Module 02) and project along $z$.

In [ ]:
# ── 2.1 Projections of β-galactosidase in random orientations ──
def project(vol: np.ndarray, R: np.ndarray) -> np.ndarray:
    """Projection along z (the beam) of the volume rotated by R: the projection approximation V_p."""
    return rotate_volume(vol, R).sum(axis=0)


t0 = time.time()
views = [project(betagal, random_rotation(rng)) for _ in range(8)]
print(f"8 projections of a 100³ map in {time.time() - t0:.1f} s")
fig, axes = plt.subplots(1, 8, figsize=(11, 1.8))
for ax, v in zip(axes, views):
    ax.imshow(v); ax.axis("off")
fig.suptitle("the same molecule in eight random orientations — what single-particle images would show without noise",
             fontsize=9)
plt.tight_layout(); plt.show()

💡 Recovering the 3D map from such views requires knowing each particle's orientation — the central computational
problem of single-particle cryo-EM. The **projection-slice theorem** (Module 07) says that each 2D projection fills one
central plane of the 3D Fourier transform; enough orientations fill all of Fourier space.

---
## 3 · The electron wavelength

Electrons accelerated through a voltage $V$ behave as waves with wavelength $\lambda = h/p$ (de Broglie). At 300 kV
they move at ≈ 78 % of the speed of light, so we need the **relativistic** momentum:

$$\lambda = \frac{h}{\sqrt{2m_0eV\left(1+\dfrac{eV}{2m_0c^2}\right)}}
  \;\approx\; \frac{12.2643}{\sqrt{V\,(1+0.978466\times10^{-6}\,V)}}\ \text{Å}\quad (V \text{ in volts}).$$

*In words:* higher voltage → faster electrons → shorter wavelength; the bracket is the relativistic correction. The
**interaction constant** $\sigma = \frac{2\pi}{\lambda V}\,\frac{m_0c^2+eV}{2m_0c^2+eV}$ (rad V⁻¹ Å⁻¹) says how much phase
shift one volt·ångström of projected potential produces; it *decreases* with voltage (faster electrons interact less —
less contrast, but also less damage per useful signal and better penetration of thick samples).

In [ ]:
# ── 3.1 Wavelength and interaction constant at common voltages ──
def interaction_constant(kV: float) -> float:
    """σ in rad / (V·Å): phase shift per unit projected potential (relativistic)."""
    V, mc2 = kV * 1e3, 510998.95                       # electron rest energy m0c² in eV
    return 2 * np.pi / (electron_wavelength_A(kV) * V) * (mc2 + V) / (2 * mc2 + V)


print(f"{'voltage':>8s} {'λ (Å)':>8s} {'non-relativistic λ':>19s} {'σ (mrad/(V·Å))':>15s}")
for kV in [100, 200, 300]:
    lam_nr = 12.2643 / np.sqrt(kV * 1e3)
    print(f"{kV:5d} kV {electron_wavelength_A(kV):8.4f} {lam_nr:19.4f} {1e3 * interaction_constant(kV):15.3f}")
assert abs(electron_wavelength_A(300) - 0.0197) < 1e-4, "λ at 300 kV should be 0.0197 Å"
print(f"visible light (green, 5,500 Å) is {5500 / electron_wavelength_A(300):,.0f}× longer than a 300 kV electron wave")

🔍 **What just happened?** At 300 kV, λ = 0.0197 Å — some 280,000× shorter than green light and far smaller than an
atom. So the wavelength is *not* what limits resolution in cryo-EM. The limits are the lens **aberrations** (next
section), **radiation damage** (Section 6) and specimen motion and thickness.

---
## 4 · The contrast transfer function (CTF)

### 4.1 Defocus turns phase into contrast

Lens imperfections and deliberate **defocus** delay the electron waves scattered to spatial frequency $k$ (in Å⁻¹;
Module 03) by an extra phase, the **wave aberration**

$$\chi(k) = \pi\lambda\,\Delta f\,k^{2} \;-\; \frac{\pi}{2}\,C_s\,\lambda^{3}k^{4},$$

where $\Delta f$ is the **defocus** (here **positive = underfocus**, the way cryo-EM images are taken) and $C_s$ the
**spherical aberration** of the objective lens (≈ 2–3 mm). Interference of scattered and unscattered waves then turns
the invisible phase shift into intensity contrast. To first order, the image is the projected potential **filtered** by
the CTF:

$$I(\mathbf r) \approx 1 + 2\sigma\,\mathcal F^{-1}\!\big[\mathrm{CTF}(k)\,\hat V_p(\mathbf k)\big],\qquad
  \mathrm{CTF}(k) = -\Big[\sqrt{1-A^2}\,\sin\chi(k) + A\cos\chi(k)\Big]\,E(k).$$

*In words:* each spatial frequency of the specimen is multiplied by a number between −1 and +1 that oscillates with $k$.
The small **amplitude contrast** $A$ (≈ 0.07 for ice-embedded proteins) adds a little contrast at the lowest frequencies;
equivalently $\mathrm{CTF}=-\sin(\chi+\arcsin A)\,E$. The **envelope** $E(k)=e^{-Bk^2/4}$ (a "B-factor", Å²) summarises
everything that damps high frequencies (partial coherence, drift, detector).

> 📐 **Sign convention used in this course** (toolkit `ctf_2d`): underfocus $\Delta f>0$, $\chi$ as above, and the
> overall minus sign, so that $\mathrm{CTF}(0)=-A<0$ and the low-frequency CTF is negative for underfocus: **protein
> appears darker than the surrounding ice**, as in raw micrographs. Software packages and papers use different signs (many
> papers write underfocus as a *negative* defocus, e.g. "−4 µm"); only the *relative* signs matter for correction.

In [ ]:
# ── 4.1 The CTF in 1D: effect of defocus, C_s and amplitude contrast ──
def ctf_1d(k: np.ndarray, defocus_um: float, kV: float = 300.0, Cs_mm: float = 2.7, amp_contrast: float = 0.07,
           B: float = 0.0) -> np.ndarray:
    """Radial CTF (same formula and sign convention as the toolkit's ctf_2d); k in 1/Å."""
    lam = electron_wavelength_A(kV)
    chi = np.pi * lam * defocus_um * 1e4 * k ** 2 - 0.5 * np.pi * Cs_mm * 1e7 * lam ** 3 * k ** 4
    return -(np.sqrt(1 - amp_contrast ** 2) * np.sin(chi) + amp_contrast * np.cos(chi)) * np.exp(-B * k ** 2 / 4)


# check against the 2D toolkit version along the x axis of a 2D grid
k_axis = np.fft.fftfreq(256)[:128] / PIX
assert np.allclose(ctf_1d(k_axis, 2.0, B=50), ctf_2d((256, 256), PIX, defocus_um=2.0, B=50)[0, :128], atol=1e-5)

k = np.linspace(0, 0.3, 2000)                           # 1/Å, up to 3.3 Å resolution
lam300 = electron_wavelength_A(300)
fig, axes = plt.subplots(3, 1, figsize=(8, 6.2), sharex=True)
for ax, df in zip(axes, [0.5, 1.5, 3.0]):
    c = ctf_1d(k, df)
    ax.plot(k, c, lw=1)
    ax.plot(k, ctf_1d(k, df, B=80), lw=1.5, color="tab:red", label="with envelope B = 80 Å²")
    k1 = 1 / np.sqrt(lam300 * df * 1e4)
    first_zero = k[1:][np.where(np.diff(np.sign(c)))[0][0]]
    ax.axvline(k1, color="gray", ls=":", lw=1)
    ax.set_ylabel("CTF"); ax.set_ylim(-1.05, 1.05); ax.axhline(0, color="k", lw=0.5)
    ax.set_title(f"Δf = {df} µm underfocus: first zero at {first_zero:.4f} Å⁻¹ ({1 / first_zero:.1f} Å); "
                 f"estimate 1/√(λΔf) = {k1:.4f} Å⁻¹", fontsize=8)
axes[0].legend(fontsize=7, loc="lower left")
axes[-1].set_xlabel("spatial frequency k (1/Å)   — resolution = 1/k")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The CTF starts slightly negative (amplitude contrast), swings to −1, then oscillates between
−1 and +1 faster and faster. Three consequences shape all of cryo-EM:

1. **Zeros**: at each zero crossing that frequency is *not transferred at all* — information lost from this image.
2. **Sign flips**: between zeros the contrast is inverted — features at those frequencies appear with reversed sign.
3. **Defocus trade-off**: more defocus gives strong low-frequency contrast (particles easy to *see*) but the first zero
   comes earlier (≈ $1/\sqrt{\lambda\Delta f}$, dotted line) and oscillations are denser. Single-particle cryo-EM uses
   ≈ 0.5–2.5 µm; cellular cryo-ET often 2–6 µm to see molecules in thick, crowded samples.

The envelope (red) shows why high-resolution information is fragile: it is damped even where the CTF is not zero.

In [ ]:
# ── 4.2 The 2D CTF and what it does to an image ──
view = np.pad(views[0], 14)                                   # one projection in a 128 × 128 box (453 Å)
defoci = [0.0, 0.5, 1.5, 3.0]
ctfs_demo = [ctf_2d(view.shape, PIX, defocus_um=df, B=50) for df in defoci]
imgs_demo = [apply_fourier_mask(view, c2) for c2 in ctfs_demo]
lim = max(np.abs(im).max() for im in imgs_demo)                 # one common grey scale for all four images
fig, axes = plt.subplots(2, len(defoci), figsize=(9, 4.8))
for j, (df, c2, img) in enumerate(zip(defoci, ctfs_demo, imgs_demo)):
    axes[0, j].imshow(np.fft.fftshift(c2), cmap="RdBu_r", vmin=-1, vmax=1)
    axes[0, j].set_title(f"2D CTF, Δf = {df} µm", fontsize=8)
    axes[1, j].imshow(img, vmin=-lim, vmax=lim)
    axes[1, j].set_title(f"image (r with V_p = {pearson(img, view):+.2f})", fontsize=8)
for ax in axes.ravel():
    ax.axis("off")
fig.suptitle("top: CTF (red +, blue −, centre = zero frequency); bottom: the projection seen through it (same grey scale)", fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** At $\Delta f = 0$ only the weak amplitude contrast is left: the particle is a faint dark
shadow (a perfect negative copy, $r=-1$, but with ≈ 7 % of the strength). With underfocus the phase contrast switches on:
the particle becomes strongly **dark** (negative correlation with $V_p$) and surrounded by **fringes** (halos) — the
"ringy" look of real micrographs. The correlation with $V_p$ drops as defocus grows because more frequency bands are
sign-flipped. The CTF pattern is a set of concentric rings: the 2D version of the 1D curves above.

✏️ **Your turn.** In cell 4.1, set `Cs_mm=0.001` (a $C_s$-corrected microscope) for the 3 µm curve, or change the
voltage to 200 kV. Which part of the curve changes?

---
## 5 · Thon rings and defocus estimation

Every micrograph carries its own CTF, so the first step of every cryo-EM pipeline is to **measure** it. The trick: the
power spectrum $|\hat I(\mathbf k)|^2$ of a micrograph is roughly the CTF² times the specimen's (fairly flat) spectrum, so
the CTF's zeros appear as dark **Thon rings**. Fitting a model CTF to these rings gives the defocus.

🧪 **Simulation.** A 1024 × 1024 micrograph at 3.54 Å/px (≈ 0.36 × 0.36 µm) containing 36 β-galactosidase particles in
random orientations, embedded in **amorphous ice** (modelled as a faint random texture of the potential, which is what
makes Thon rings visible even between particles). We filter with the CTF (Δf = 2.0 µm, 300 kV, $C_s$ = 2.7 mm, A = 0.07,
B = 80 Å²) and then count electrons with a dose of 40 e⁻/Å² (Section 6 explains the noise). The contrast scale is chosen
so that the protein changes the beam intensity by at most ≈ 10 % (an assumption of this toy).

In [ ]:
# ── 5.1 Simulate a micrograph ──
def place_particles(vol: np.ndarray, n: int, size: int, rng, min_dist: float) -> tuple[np.ndarray, list]:
    """Sum projections of `vol` (random orientations) at random non-overlapping positions in a size×size image."""
    img, centres, half = np.zeros((size, size), np.float32), [], vol.shape[1] // 2
    for _ in range(20_000):
        if len(centres) == n:
            break
        c = rng.uniform(half, size - half, 2)
        if all(np.hypot(*(c - d)) > min_dist for d in centres):
            centres.append(c)
            y0, x0 = int(c[0]) - half, int(c[1]) - half
            img[y0:y0 + 2 * half, x0:x0 + 2 * half] += project(vol, random_rotation(rng))
    return img, centres


def expose(contrast_image: np.ndarray, dose_e_per_A2: float, pix: float, rng) -> np.ndarray:
    """Electron counting: Poisson counts with mean dose·pixel_area·(1 + contrast) per pixel."""
    return rng.poisson(dose_e_per_A2 * pix ** 2 * np.clip(1 + contrast_image, 0, None)).astype(np.float32)


rng = np.random.default_rng(SEED + 5)
t0 = time.time()
MIC, DF_TRUE, CONTRAST, DOSE = 1024, 2.0, 0.25, 40.0
potential, centres = place_particles(betagal, 36, MIC, rng, min_dist=100)
potential /= potential.max()                                            # projected potential, max = 1
ice = ndimage.gaussian_filter(rng.normal(size=(MIC, MIC)), 0.7)
potential_with_ice = potential + 0.08 * ice / ice.std()
ctf_mic = ctf_2d((MIC, MIC), PIX, defocus_um=DF_TRUE, kV=300, Cs_mm=2.7, amp_contrast=0.07, B=80)
contrast_img = CONTRAST * apply_fourier_mask(potential_with_ice, ctf_mic)   # (I − 1): at most ≈ 10 %
micrograph = expose(contrast_img, DOSE, PIX, rng)
signal_var = np.var(DOSE * PIX ** 2 * CONTRAST * apply_fourier_mask(potential, ctf_mic))
print(f"{len(centres)} particles; mean {micrograph.mean():.0f} electrons/pixel; contrast range "
      f"{100 * contrast_img.min():+.1f} … {100 * contrast_img.max():+.1f} %; per-pixel SNR of the protein signal ≈ "
      f"{signal_var / (DOSE * PIX ** 2):.3f}  ({time.time() - t0:.1f} s)")

crop = slice(0, 384)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.9))
axes[0].imshow(potential[crop, crop]); axes[0].set_title("projected potential (truth)", fontsize=9)
axes[1].imshow(micrograph[crop, crop]); axes[1].set_title(f"micrograph: {DOSE:.0f} e⁻/Å², Δf {DF_TRUE} µm", fontsize=9)
axes[2].imshow(ndimage.gaussian_filter(micrograph[crop, crop], 2.5)); axes[2].set_title("same, low-pass filtered for display", fontsize=9)
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
axes[0].add_patch(plt.Rectangle((15, 360), 500 / PIX, 6, color="w")); axes[0].text(15, 350, "50 nm", color="w", fontsize=8)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The raw micrograph looks like grey static: the per-pixel SNR is far below 1. Only after
low-pass filtering do dark particles with light halos emerge. The bigger the particle, the easier it is to see — one reason
small proteins (< 100 kDa) remain hard for cryo-EM.

In [ ]:
# ── 5.2 Power spectrum, Thon rings and a defocus fit ──
def tile_power_spectrum(img: np.ndarray, tile: int = 128) -> np.ndarray:
    """Average |FFT|² of non-overlapping tiles (mean removed) — a smooth estimate of the power spectrum."""
    tiles = [img[i:i + tile, j:j + tile] for i in range(0, img.shape[0] - tile + 1, tile)
             for j in range(0, img.shape[1] - tile + 1, tile)]
    return np.mean([np.abs(np.fft.fft2(t - t.mean())) ** 2 for t in tiles], axis=0)


def radial_mean(ps: np.ndarray, pix: float) -> tuple[np.ndarray, np.ndarray]:
    """Rotational average of an (unshifted) power spectrum. Returns (k in 1/Å, mean power per ring)."""
    n = ps.shape[0]
    ring = np.minimum((radial_frequency(ps.shape) * n).round().astype(int), n // 2)
    prof = np.bincount(ring.ravel(), ps.ravel(), n // 2 + 1) / np.bincount(ring.ravel(), minlength=n // 2 + 1)
    return np.arange(n // 2 + 1) / (n * pix), prof


def estimate_defocus(k: np.ndarray, prof: np.ndarray, grid_um: np.ndarray, k_min: float = 1 / 40, k_max: float = 1 / 8):
    """Grid search: correlate the background-subtracted log power spectrum with CTF²(k; Δf). Returns (best Δf, scores)."""
    logp = np.log(prof + 1e-12)
    flat = logp - ndimage.uniform_filter1d(logp, 9)            # remove the smooth background (ice, noise, envelope)
    sel = (k >= k_min) & (k <= k_max)
    scores = np.array([pearson(flat[sel], ctf_1d(k[sel], d) ** 2) for d in grid_um])
    return grid_um[np.argmax(scores)], scores


ps = tile_power_spectrum(micrograph)
k_r, prof = radial_mean(ps, PIX)
grid = np.arange(0.5, 4.0, 0.01)
df_est, scores = estimate_defocus(k_r, prof, grid)
print(f"true defocus {DF_TRUE:.2f} µm → estimated {df_est:.2f} µm (from {MIC // 128}² = {(MIC // 128) ** 2} tiles of 128²)")
assert abs(df_est - DF_TRUE) < 0.1, "the defocus fit should land within 0.1 µm"

fig = plt.figure(figsize=(11, 3.4))
ax = fig.add_subplot(1, 3, 1)
half = np.fft.fftshift(np.log(ps + 1e-6))
theory = np.fft.fftshift(ctf_2d(ps.shape, PIX, defocus_um=df_est, B=80) ** 2)
combo = half.copy(); combo[:, 64:] = (theory[:, 64:] - theory.mean()) / theory.std() * half.std() + half.mean()
ax.imshow(combo, vmin=np.percentile(half, 1), vmax=np.percentile(half, 99.5)); ax.axis("off")
ax.set_title("power spectrum: data (left) | fitted CTF² (right)", fontsize=8)
ax = fig.add_subplot(1, 3, 2)
sel = (k_r > 0.005) & (k_r < 0.14)
ax.plot(k_r[sel], np.log(prof[sel]), label="rotational average")
for z in k_r[1:][np.where(np.diff(np.sign(ctf_1d(k_r, df_est))))[0]]:
    if z < 0.14:
        ax.axvline(z, color="tab:red", lw=0.6)
ax.set_xlabel("k (1/Å)"); ax.set_ylabel("log power"); ax.set_title("Thon rings; red: fitted CTF zeros", fontsize=8)
ax = fig.add_subplot(1, 3, 3)
ax.plot(grid, scores); ax.axvline(DF_TRUE, color="gray", ls=":", label="truth")
ax.set_xlabel("trial defocus (µm)"); ax.set_ylabel("fit score (correlation)"); ax.legend(fontsize=7)
ax.set_title(f"best Δf = {df_est:.2f} µm", fontsize=8)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Averaging the power spectra of 64 tiles made the Thon rings stand out of the counting noise.
The rotational average shows dips exactly where the fitted CTF has zeros, and the fit score peaks at the true defocus.
This is the core of CTF-estimation programs such as CTFFIND and Gctf — which also fit **astigmatism** (elliptical rings,
two defocus values), phase shifts and envelopes. In cryo-ET, every tilt image needs its own fit, with less signal per
image and a **defocus gradient** across the tilted specimen.

✏️ **Your turn.** Set `DOSE = 5.0` (a single tilt image of a tomogram is ≈ 2–3 e⁻/Å²) and re-run 5.1–5.2. Does the
fit still work? Then try `DF_TRUE = 4.0`.

### 📄 Paper connection — microscope settings inside the Xu-lab simulators

Many Xu-lab methods are developed and benchmarked on **simulated** tomograms whose image formation follows exactly the
chain of this notebook (projection → CTF → noise → reconstruction). The settings reported in the papers:

| Simulation (paper) | Voltage | $C_s$ | Defocus (as written) | Other | Module |
|---|---|---|---|---|---|
| Template-free detection — Xu, Beck & Alber 2011, *Bioinformatics* 27:i69 ([doi:10.1093/bioinformatics/btr207](https://doi.org/10.1093/bioinformatics/btr207)) | 200 kV | 2 mm | −4 µm | tilts ±70° in 2° steps; CTF and MTF sinc(πω/2); 1 nm voxels, 4 nm resolution | 14 |
| Fast alignment & classification — Xu, Beck & Alber 2012, *J Struct Biol* 178:152 ([doi:10.1016/j.jsb.2012.02.014](https://doi.org/10.1016/j.jsb.2012.02.014)) | 200 kV | 2 mm | −4 µm | MTF sinc(πω/2); 1 nm voxels; SNR 0.01, ±60° | 11, 13 |
| Crowded-cell simulation — Pei *et al.* 2016, *BMC Bioinformatics* 17:405 ([doi:10.1186/s12859-016-1283-3](https://doi.org/10.1186/s12859-016-1283-3)) | 300 kV | 2 mm | −4 µm | ±60° in 2° steps; noise added before and after the CTF; back-projection | 08, 10 |
| Shared deep-learning benchmark (Xu *et al.* 2017 and later) | 300 kV | 2 mm | −5 µm | 40³ subtomograms, 0.92 nm voxels; EMAN2 reconstruction | 15 |
| SHREC'20 benchmark (Gubins *et al.* 2020, *Comput Graph* 91:279) | 300 kV | — | 2–5 µm | 61 tilts over ±60°; dose 100–120 e⁻/Å² | 05, 15 |

- **Sign convention.** "Defocus −4 µm" in these papers is a 4 µm **underfocus** written with the opposite sign
  convention; in this course's convention it is $\Delta f = +4$ µm (`ctf_2d(..., defocus_um=4.0)`).
- **MTF** (modulation transfer function) describes how the *detector* blurs the image; the papers model it as
  $\mathrm{sinc}(\pi\omega/2)$ (as written in the papers; we omit it in our toy).
- **What our toy keeps / leaves out.** We keep projection, CTF (same formula), envelope and noise. We leave out the MTF,
  per-tilt geometry and reconstruction (Module 07), and simulators' full noise models.
- **Code.** AITom `aitom/simulation/` (tomogram simulation; Pei *et al.* 2016 packing in
  `aitom/simulation/tomogram/single_bounding_sphere/`).

---
## 6 · Dose, shot noise and radiation damage

### 6.1 Counting electrons

A detector pixel counts individual electrons. If on average $\mu$ electrons arrive, the actual count follows a
**Poisson distribution**: $P(n) = \mu^n e^{-\mu}/n!$, with **variance equal to the mean**, $\operatorname{var}(n)=\mu$.
The expected count per pixel is

$$\mu = D \cdot p^{2} \cdot (1 + \text{contrast}),$$

with **dose** $D$ (e⁻/Å²) and pixel size $p$ (Å). The signal (contrast × $Dp^2$) grows like $D$; the noise std like
$\sqrt{D p^2}$. *In words:* **SNR per pixel ∝ dose** — ten times more electrons, ten times better SNR (the √N law of
Module 04, with electrons as the "copies").

In [ ]:
# ── 6.1 The same field of particles at three doses ──
rng = np.random.default_rng(SEED + 6)
field = contrast_img[:256, :256]                                  # CTF-filtered contrast of a 256² patch
protein_only = CONTRAST * apply_fourier_mask(potential, ctf_mic)[:256, :256]
doses = [1.0, 10.0, 100.0]
fig, axes = plt.subplots(1, 4, figsize=(11.5, 2.9))
for ax, d in zip(axes, doses):
    counts = expose(field, d, PIX, rng)
    snr = np.var(d * PIX ** 2 * protein_only) / (d * PIX ** 2)
    ax.imshow(ndimage.gaussian_filter(counts, 1.5)); ax.axis("off")
    ax.set_title(f"{d:g} e⁻/Å² → {d * PIX ** 2:.0f} e⁻/px\nper-pixel SNR ≈ {snr:.4f}", fontsize=8)
mu = 40.0 * PIX ** 2
draws = rng.poisson(mu, 20_000)
axes[3].hist(draws, bins=60, density=True, color="lightgray", edgecolor="gray")
xx = np.arange(draws.min(), draws.max())
axes[3].plot(xx, np.exp(-(xx - mu) ** 2 / (2 * mu)) / np.sqrt(2 * np.pi * mu), "tab:red", label="Gaussian, var = mean")
axes[3].set_title(f"Poisson counts, mean {mu:.0f}:\nvariance {draws.var():.0f}", fontsize=8); axes[3].legend(fontsize=6)
axes[3].set_xlabel("electrons per pixel")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The per-pixel SNR grows in proportion to the dose (1 → 10 → 100 e⁻/Å²), and the Poisson
counts have variance ≈ mean, looking Gaussian at these count levels (which is why Gaussian noise models are common).
A single **tilt image** of a tomogram receives only ≈ 2–3 e⁻/Å² (between the left and middle panels); a
single-particle micrograph ≈ 40–60 e⁻/Å² (spread over a movie of many frames). The SNR values are tiny because they are
averaged over the whole field, most of which is ice — and because protein contrast is only a few per cent.

### 6.2 Why not use more electrons? Radiation damage

Each electron that scatters inelastically deposits energy and breaks bonds; high-resolution details are destroyed
first. Grant & Grigorieff (2015) measured the **critical exposure** $N_e(k)$ — the dose after which the signal at
frequency $k$ has decayed by a factor $e^{-1/2}$ in amplitude:

$$N_e(k) \approx 0.245\,k^{-1.665} + 2.81\ \ \text{e}^-/\text{Å}^2 \quad (k \text{ in Å}^{-1},\ 300\text{ kV}),\qquad
  \text{amplitude} \propto \exp\!\Big(-\frac{N}{2N_e(k)}\Big).$$

*In words:* 3 Å details fade after a few electrons per Å², 20 Å details survive tens. This is why modern detectors
record **movies** (dose fractionation): early frames keep high-resolution information, later frames add low-resolution
signal, and "dose weighting" combines them (see the motion-correction deep dive). In cryo-ET the total budget
(≈ 100–120 e⁻/Å² in SHREC'20's simulations; 80–100 e⁻/Å² in the fungal-membrane study of Module 33) is shared by 40–60
tilt images — which is why **dose-symmetric** tilt schemes collect the most informative low tilts first (Module 07).

In [ ]:
# ── 6.2 How much signal survives a given cumulative dose? ──
def critical_exposure(k: np.ndarray) -> np.ndarray:
    """Grant & Grigorieff (2015) critical exposure in e⁻/Å² for spatial frequency k (1/Å), 300 kV."""
    return 0.245 * k ** -1.665 + 2.81


kd = np.linspace(1 / 60, 1 / 2.5, 400)
fig, ax = plt.subplots(figsize=(6, 3.2))
for N in [5, 20, 50, 120]:
    ax.plot(1 / kd, np.exp(-N / (2 * critical_exposure(kd))), label=f"after {N} e⁻/Å²")
ax.set_xscale("log"); ax.invert_xaxis(); ax.minorticks_off()
ax.set_xticks([60, 30, 20, 10, 5, 3]); ax.set_xticklabels(["60", "30", "20", "10", "5", "3"])
ax.set_xlabel("resolution (Å)  →  finer details"); ax.set_ylabel("remaining signal amplitude")
ax.set_title("radiation damage destroys fine details first"); ax.legend(fontsize=7); plt.tight_layout(); plt.show()
for res_A in [20, 8, 4, 3]:
    print(f"details at {res_A:2d} Å: critical exposure ≈ {critical_exposure(1 / res_A):5.1f} e⁻/Å²")

🔍 **What just happened?** After 20 e⁻/Å² almost nothing remains at 3 Å while 20 Å features are largely intact; after
120 e⁻/Å² (a whole tilt series) only the coarse shape survives in the late images. Cryo-EM is a permanent negotiation
between **noise** (too few electrons) and **damage** (too many) — and computation is how we win it.

### 📄 Paper connection — score-based denoising of cryo-EM micrographs (ECCV 2026)

**Paper.** *Score-Based Matching with Target Guidance for Cryo-EM Denoising.* ECCV 2026 (listed in AITom's publication
list). ⚠️ **We could not access the full text of this paper while writing; the description below is limited to the
released code (AITom `aitom/filter/denoise/tsm/`) and its README. The author list could not be verified. Read the paper
before relying on details.**

- **Question it answers.** How to denoise 2D cryo-EM micrographs without clean/noisy training pairs.
- **What the code does.** Trains a U-Net to output a *score* (the gradient of the log-density of images) by denoising
  score matching on the noisy data themselves, plus a "target-guided" term that pulls the score toward a soft clean
  target built from a bank of clean reference projections, gated by feature similarity. Denoising then applies a few
  Tweedie-style steps $x \leftarrow x + \eta\,\sigma(x)^2 s_\theta(x)$ with a **signal-dependent** noise level
  $\sigma(x)=a+bx$ — a nod to the fact (Section 6.1) that counting noise grows with the signal.
- **Connection to this module.** The noise it must remove is exactly the shot noise and CTF-filtered contrast simulated
  here. Module 24 builds a toy version (Tweedie's formula, score matching).
- **Result reported.** Not verified (no paper text available to us).

---
## 7 · Correcting the CTF: phase flipping and Wiener filtering

The CTF damages an image in two ways: it **flips the sign** of some frequencies and **attenuates** others (to zero at
its zeros). Two classic corrections, applied in Fourier space ($\hat Y$ = Fourier transform of the observed image):

- **Phase flipping:** $\hat X = \operatorname{sign}(\mathrm{CTF})\cdot\hat Y$ — undo the sign flips only. Simple and
  safe; amplitudes stay attenuated.
- **Wiener filter:** $\hat X(\mathbf k) = \dfrac{\mathrm{CTF}(\mathbf k)}{\mathrm{CTF}(\mathbf k)^2 + 1/\mathrm{SNR}(k)}\,\hat Y(\mathbf k)$,
  where $\mathrm{SNR}(k)$ is the **spectral SNR**: signal power divided by noise power *at frequency k*. *In words:*
  divide by the CTF where the signal is strong enough to trust, give up where noise dominates. (Dividing by the CTF
  directly would explode near its zeros.)

A particle's power is concentrated at low frequencies, so $\mathrm{SNR}(k)$ is large at low $k$ and tiny at high $k$. In a
simulation we know the true signal spectrum (an "oracle"); real software estimates $\mathrm{SNR}(k)$ from the data, for
example from the Fourier shell correlation of two half-sets (Module 10). We also try the lazy version with **one constant
SNR for all frequencies**.

🧪 One particle projection, Δf = 1.5 µm, Gaussian noise at SNR 1 (a generous level, so the effects are visible).

In [ ]:
# ── 7.1 Phase flipping and Wiener filtering of one particle image ──
def phase_flip(img: np.ndarray, ctf: np.ndarray) -> np.ndarray:
    """Multiply every Fourier coefficient by the sign of the CTF."""
    return apply_fourier_mask(img, np.sign(ctf))


def wiener_filter(img: np.ndarray, ctf: np.ndarray, snr) -> np.ndarray:
    """Single-image Wiener filter CTF / (CTF² + 1/SNR); `snr` is a number or a per-frequency array (spectral SNR)."""
    return apply_fourier_mask(img, ctf / (ctf ** 2 + 1.0 / snr))


def radial_power_map(img: np.ndarray) -> np.ndarray:
    """|FFT|² averaged over rings of equal |k|, returned on the full (unshifted) FFT grid."""
    n = img.shape[0]
    ring = np.minimum((radial_frequency(img.shape) * n).round().astype(int), n // 2)
    power = np.abs(np.fft.fft2(img)) ** 2
    prof = np.bincount(ring.ravel(), power.ravel(), n // 2 + 1) / np.bincount(ring.ravel(), minlength=n // 2 + 1)
    return prof[ring]


rng = np.random.default_rng(SEED + 7)
truth = np.pad(views[1], 14)                                    # 128 × 128 projection
ctf_p = ctf_2d(truth.shape, PIX, defocus_um=1.5, B=50)
clean_obs = apply_fourier_mask(truth, ctf_p)
observed = add_noise_snr(clean_obs, 1.0, rng)
noise_var = clean_obs.var() / 1.0                              # what add_noise_snr used
ssnr = radial_power_map(truth) / (noise_var * truth.size)      # oracle spectral SNR (white noise: power = var·N)
results = {"observed (CTF + noise)": observed, "phase-flipped": phase_flip(observed, ctf_p),
           "Wiener, one constant SNR": wiener_filter(observed, ctf_p, snr=1.0),
           "Wiener, spectral SNR(k)": wiener_filter(observed, ctf_p, snr=ssnr)}
fig, axes = plt.subplots(1, 5, figsize=(12, 2.9))
axes[0].imshow(truth); axes[0].set_title("truth V_p", fontsize=9)
for ax, (name, img) in zip(axes[1:], results.items()):
    ax.imshow(img); ax.set_title(f"{name}\nr with truth = {pearson(img, truth):+.2f}", fontsize=8)
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()

kk = np.linspace(0, 1 / (2 * PIX), 600)
c1 = ctf_1d(kk, 1.5, B=50)
fig, ax = plt.subplots(figsize=(7, 2.8))
ax.plot(kk, c1, lw=1, color="gray", label="CTF (what the microscope did)")
ax.plot(kk, np.sign(c1) * c1, lw=1.5, label="after phase flipping: |CTF|")
ssnr_1d = np.interp(kk, np.fft.fftfreq(128)[:64] / PIX, ssnr[0, :64])
ax.plot(kk, c1 ** 2 / (c1 ** 2 + 1.0), lw=1, ls="--", label="after Wiener, constant SNR 1: CTF²/(CTF²+1)")
ax.plot(kk, c1 ** 2 / (c1 ** 2 + 1 / ssnr_1d), lw=1.5, label="after Wiener, spectral SNR(k)")
ax.axhline(0, color="k", lw=0.5); ax.set_xlabel("k (1/Å)"); ax.set_ylabel("net transfer (relative to truth)")
ax.set_title("net transfer of each frequency after correction"); ax.legend(fontsize=6); plt.tight_layout(); plt.show()

🔍 **What just happened?** The observed image correlates *negatively* with the truth (dark protein, flipped bands).
Phase flipping makes every frequency's net transfer positive — the particle becomes bright and the correlation positive,
but the overall shape stays faint because the CTF had almost erased the lowest frequencies (only the 7 % amplitude
contrast transfers them). The **Wiener filter with the spectral SNR** is dramatically better: at low frequencies, where
the particle's signal is strong, it divides the CTF out and restores the particle's shape; at high frequencies, where
noise dominates, it gives up. The *same formula with one constant SNR* does worse than phase flipping — **a Wiener filter
is only as good as its noise model**. And no filter can recover frequencies at the CTF zeros of a single image (here they
carry little of the particle's power, so the correlation barely notices).

✏️ **Your turn.** Lower the noise to SNR 0.1: change `add_noise_snr(clean_obs, 1.0, rng)` and `noise_var = clean_obs.var() / 1.0`
to 0.1 and re-run. Which correction suffers most?

---
## 8 · Pixel size, Nyquist and binning

A pixel of size $p$ can represent spatial frequencies only up to the **Nyquist frequency** $k_N = 1/(2p)$ — the finest
detail is two pixels per period, i.e. a best possible resolution of $2p$ (Module 03). **Binning** by $b$ (averaging
$b\times b$ blocks) makes pixels $b$ times larger: images become smaller and less noisy per pixel, computations cheaper,
but everything beyond the new Nyquist frequency is lost. Cellular tomograms are usually binned to 8–14 Å voxels
(the zoo maps of this course use 10 Å; DISCA's subtomograms 13.33 Å) — so they "see" only the first few CTF lobes.

In [ ]:
# ── 8.1 Binning a particle image, and where Nyquist cuts the CTF ──
def bin_image(img: np.ndarray, b: int) -> np.ndarray:
    """Average non-overlapping b×b blocks (image size must be divisible by b)."""
    h, w = img.shape
    return img.reshape(h // b, b, w // b, b).mean(axis=(1, 3))


fig = plt.figure(figsize=(11, 3.2))
for i, b in enumerate([1, 2, 4]):
    ax = fig.add_subplot(1, 5, i + 1)
    ax.imshow(bin_image(truth, b)); ax.axis("off")
    ax.set_title(f"bin {b}: {PIX * b:.2f} Å/px\nNyquist {2 * PIX * b:.1f} Å", fontsize=8)
ax = fig.add_subplot(1, 5, (4, 5))
kk = np.linspace(0.001, 0.5, 3000)
ax.plot(kk, ctf_1d(kk, 4.0, B=50), lw=0.8, color="gray", label="CTF, Δf = 4 µm (cryo-ET-like)")
for p, col, lab in [(1.0, "k", "detector ≈ 1 Å/px"), (PIX, "tab:blue", f"{PIX} Å/px (this map)"),
                    (10.0, "tab:green", "10 Å voxels (zoo)"), (13.33, "tab:red", "13.33 Å (DISCA)")]:
    ax.axvline(1 / (2 * p), color=col, lw=1.5, label=f"Nyquist at {lab}")
ax.set_xscale("log"); ax.set_xlabel("k (1/Å), log scale"); ax.set_ylabel("CTF"); ax.legend(fontsize=6, loc="lower left")
ax.set_title("everything right of a line is lost at that pixel size", fontsize=8)
plt.tight_layout(); plt.show()
for p in [1.0, PIX, 10.0, 13.33]:
    zeros = np.sum(np.diff(np.sign(ctf_1d(kk[kk <= 1 / (2 * p)], 4.0))) != 0)
    print(f"pixel {p:5.2f} Å → Nyquist {2 * p:5.1f} Å; CTF zero crossings below Nyquist at Δf = 4 µm: {zeros}")

🔍 **What just happened?** Binning by 4 (14 Å pixels) keeps the overall shape of β-galactosidase but loses its
internal detail. At 10–13 Å voxels a 4 µm-defocus tomogram contains just the first CTF zero — so for picking
and classification in binned tomograms the CTF matters mostly through its low-frequency behaviour (contrast inversion,
the first zero), while subtomogram averaging toward high resolution must correct it fully per tilt.

---
## 9 · Single-particle cryo-EM vs cryo-ET, and 2D class averages

| | **Single-particle cryo-EM (SPA)** | **Cellular cryo-ET** |
|---|---|---|
| Specimen | purified molecules in ≈ 20–100 nm ice | whole small cells or FIB-milled lamellae, ≈ 100–300 nm |
| Images per specimen area | one micrograph (a movie of many frames) | a tilt series of ≈ 40–60 images over ±60° |
| Dose | ≈ 40–60 e⁻/Å² per micrograph | ≈ 80–120 e⁻/Å² **in total**, ≈ 2–3 e⁻/Å² per tilt |
| Typical defocus | ≈ 0.5–2.5 µm | ≈ 2–6 µm (more contrast for thick, crowded samples) |
| CTF handling | per micrograph (with astigmatism) | per tilt image, with a defocus gradient across the tilted specimen |
| How 3D is obtained | combine *different* particles in random orientations | reconstruct *one* specimen from its tilts (Module 07) |
| Missing information | none systematic (if orientations are diverse) | the **missing wedge** beyond ±60° |
| Typical result | near-atomic maps of purified complexes | molecular landscapes in situ; subtomogram averages (≈ 3–30 Å) |

### 9.1 2D class averaging across many defoci

A **2D class average** combines many aligned images of the same view (Module 04: SNR grows ∝ N). Because every
micrograph has its own defocus, each image has its own CTF. Simply averaging raw images mixes CTFs whose bands have
*opposite signs* at many frequencies, which partly cancel. The fix is a **multi-image Wiener filter**:

$$\hat X(\mathbf k) = \frac{\sum_i \mathrm{CTF}_i(\mathbf k)\,\hat Y_i(\mathbf k)}{\sum_i \mathrm{CTF}_i(\mathbf k)^2 + 1/\mathrm{SNR}(k)} ,$$

with $\mathrm{SNR}(k)$ the spectral SNR of one image (oracle here, as in Section 7).

*In words:* weight each image's frequency by how well its own CTF transmitted it. Because different defoci have zeros at
different places, $\sum_i\mathrm{CTF}_i^2$ has **no zeros** — the gaps of one image are filled by the others. This is why
data are collected over a **range of defoci**.

In [ ]:
# ── 9.1 Averaging 200 noisy images of one view taken at different defoci ──
rng = np.random.default_rng(SEED + 9)
n_img, snr_img = 200, 0.05
dfs = rng.uniform(0.8, 2.5, n_img)
ctfs = np.stack([ctf_2d(truth.shape, PIX, defocus_um=d, B=50) for d in dfs])
clean_imgs = [apply_fourier_mask(truth, c) for c in ctfs]
F_obs = np.stack([np.fft.fft2(add_noise_snr(ci, snr_img, rng)) for ci in clean_imgs])
ssnr_img = radial_power_map(truth) / (np.mean([ci.var() for ci in clean_imgs]) / snr_img * truth.size)
naive = np.real(np.fft.ifft2(F_obs.mean(0)))
flipped = np.real(np.fft.ifft2((np.sign(ctfs) * F_obs).mean(0)))
wiener_avg = np.real(np.fft.ifft2((ctfs * F_obs).sum(0) / ((ctfs ** 2).sum(0) + 1.0 / ssnr_img)))
fig, axes = plt.subplots(1, 5, figsize=(12, 2.8))
for ax, (img, t) in zip(axes[:4], [(truth, "truth"), (naive, "naive average"),
                                   (flipped, "phase-flipped average"), (wiener_avg, "Wiener average")]):
    ax.imshow(img); ax.axis("off"); ax.set_title(f"{t}\nr = {pearson(img, truth):+.2f}", fontsize=8)
kk = np.linspace(0, 1 / (2 * PIX), 600)
axes[4].plot(kk, ctf_1d(kk, 1.5, B=50) ** 2, lw=1, color="gray", label="one image: CTF²")
axes[4].plot(kk, np.mean([ctf_1d(kk, d, B=50) ** 2 for d in dfs], axis=0), lw=2, label="mean over 200 defoci")
axes[4].set_xlabel("k (1/Å)"); axes[4].legend(fontsize=6); axes[4].set_title("zeros are filled in", fontsize=8)
plt.tight_layout(); plt.show()
print("correlation with truth — naive {:.2f}, phase-flipped {:.2f}, Wiener (spectral SNR) {:.2f}".format(
    pearson(naive, truth), pearson(flipped, truth), pearson(wiener_avg, truth)))

🔍 **What just happened?** Each image alone has SNR 0.05. The naive average keeps the low-frequency contrast (all CTFs
are negative there — dark particle) but blurs the bands that change sign between defoci. Phase flipping before averaging
recovers a bright, sharper particle; the multi-image Wiener average weights every frequency by how well each image
transmitted it and restores the low frequencies too — it is essentially perfect here (with an oracle SNR model). The
right panel shows why combining defoci works: the mean CTF² over many defoci never touches zero.

In real data the images must first be **aligned** (Module 04 showed what misalignment does), **classified** (different
views and different molecules), and **selected** (bad particles hurt). These are the topics of Modules 10, 12 and 14.

### 📄 Paper connection — CryoMAE: few-shot particle picking in micrographs

**Paper.** Xu C, Zhan X, Xu M (2025). *CryoMAE: Few-shot cryo-EM particle picking with masked autoencoders.* **WACV
2025**, pp. 3876–3885. [doi:10.1109/WACV61041.2025.00381](https://doi.org/10.1109/WACV61041.2025.00381); arXiv:2404.10178.

- **Question it answers.** Picking particles in single-particle micrographs like the one in Section 5 is hard at low
  SNR, and supervised pickers need many labels. Can a handful of examples suffice?
- **What the paper actually does.** A vision-transformer **masked autoencoder** is trained on ≈ 15 labelled particle
  crops (exemplars) plus random unlabelled crops, with an extra *self-cross similarity* loss that pulls particle embeddings
  together and away from background. The encoder then slides over query micrographs; each window is scored by its maximum
  cosine similarity to the exemplars, and a per-micrograph threshold is set at the elbow of the k-nearest-neighbour distances.
- **What this module gives you for it.** What a micrograph *is*: CTF-filtered, dose-limited projections in ice —
  dark particles with halos, per-pixel SNR ≪ 1.
- **Result reported.** On five CryoPPP/EMPIAR datasets the authors report an average F1 of 0.591 vs 0.502 for crYOLO
  and 0.394 for Topaz, and an average resolution of 10.13 Å after reconstruction (vs 11.40 Å Topaz, 12.10 Å crYOLO).
- **Code.** The paper points to the AITom repository; a dedicated CryoMAE folder was not found there by our notes.
  Module 31 builds a tiny masked-autoencoder picker.

### 📄 Paper connection (preview) — selecting homogeneous images to maximise resolution

**Paper.** Shi J, Zeng X, Jiang R, Jiang T, Xu M (2020). *A simulated annealing approach for resolution guided homogeneous
cryo-electron microscopy image selection.* **Quantitative Biology** 8(1):51–63. [doi:10.1007/s40484-019-0191-8](https://doi.org/10.1007/s40484-019-0191-8)

- **Question it answers.** In a pile of noisy 2D images (or subtomograms), only some show the same structure in the same
  pose. Which subset gives the best average?
- **What the paper actually does.** Scores a subset $S$ by a resolution score $r(S)=\sum_J s(J)/(2+s(J))$ computed from
  the spectral SNR of the subset, and searches over subsets by **simulated annealing** (add/replace moves, a temperature-
  dependent acceptance of worse moves, a contraction step), with a restart variant (SA+) for 3D data.
- **Connection to this module.** Section 9 assumed all 200 images were good and aligned; real class averages are only as
  good as the images that go into them. Module 10 implements a toy version (with the standard Metropolis rule; the notes
  flag a sign discrepancy between the paper's rendering and the released code).
- **Result reported.** The authors report that SA beats greedy matching and a genetic algorithm in $F_{0.5}$ and
  resolution score on 2D data, with a growing advantage as the fraction of homogeneous images shrinks.
- **Code.** AITom `aitom/classify/select/simulated_annealing/` (Python 2.7).

---
## ✅ Checkpoint

1. Why does an in-focus image of a thin protein sample show almost no contrast, and how does defocus help?
   <details><summary>Answer</summary>The specimen is a weak phase object: it shifts the phase of the electron wave but barely changes its amplitude, and a detector records intensity |ψ|² ≈ 1. Defocus (with C_s) adds a frequency-dependent phase χ(k) to the scattered waves, so their interference with the unscattered beam converts phase into intensity — described by the CTF ≈ −sin χ.</details>
2. Compute λ at 200 kV and the approximate first CTF zero for Δf = 2 µm at 300 kV.
   <details><summary>Answer</summary>λ(200 kV) ≈ 0.0251 Å. First zero k₁ ≈ 1/√(λΔf) = 1/√(0.0197 × 20,000 Å²) ≈ 0.050 Å⁻¹ → ≈ 20 Å.</details>
3. A paper says "defocus −3 µm". In this course's convention, what is Δf and will protein look dark or bright in the raw image?
   <details><summary>Answer</summary>It is 3 µm underfocus, Δf = +3 µm here; at low frequencies the CTF is negative, so protein appears dark on a lighter background.</details>
4. Doubling the dose doubles the per-pixel SNR. Why not use 500 e⁻/Å²?
   <details><summary>Answer</summary>Radiation damage: high-resolution information decays after a few e⁻/Å² (critical exposure ≈ 5 e⁻/Å² at 4 Å), and even coarse features are damaged at very high doses. The dose budget is a trade-off between noise and damage.</details>
5. Why are single-particle datasets collected over a range of defoci?
   <details><summary>Answer</summary>Each defocus has CTF zeros at different frequencies; combining images (e.g., with a multi-image Wiener filter) fills every frequency because Σ CTF_i² has no zeros.</details>
6. A tomogram is binned to 13.3 Å voxels. What is the best resolution any subtomogram average computed from it can reach?
   <details><summary>Answer</summary>Nyquist: 2 × 13.3 ≈ 26.7 Å (in practice somewhat worse). Higher resolution requires re-extracting particles from less-binned data.</details>

## 🧠 Questions & discussion

1. Cellular cryo-ET uses larger defocus than single-particle cryo-EM. Argue both for and against this choice.
2. The projection approximation assumes a thin specimen. How might it fail for a 300 nm-thick lamella, and what would
   that do to the CTF model (hint: different depths are at different defoci)?
3. Our Thon-ring fit used rotational averages. Which real-world effect would make the rings elliptical, and how would you
   extend `estimate_defocus` to measure it?
4. Phase flipping is "safe" and Wiener filtering is "optimal". What does the Wiener filter need to know that phase
   flipping does not, and what happens if that knowledge is wrong?
5. In SHREC'20, simulated tilt series used 100–120 e⁻/Å² over 61 tilts. Estimate the dose per tilt and the per-pixel SNR
   of a single tilt at 10 Å pixels relative to a 40 e⁻/Å² micrograph at 1 Å pixels. Which is noisier per pixel, and why
   is the answer surprising?

## 🚀 Bonus — going further

- **Astigmatism.** Write a `ctf_2d_astig` with two defocus values along rotated axes (a new function — do not edit the
  toolkit's `ctf_2d`), simulate elliptical Thon rings, and fit them with a 3-parameter grid search.
- **Phase plates.** A Volta phase plate adds ≈ π/2 to χ; plot the CTF with `chi + np.pi / 2` and explain why in-focus
  imaging becomes possible.
- **Real micrographs.** Download a few micrographs from [EMPIAR](https://www.ebi.ac.uk/empiar/) (e.g. EMPIAR-10081, used
  by CryoMAE) and run `tile_power_spectrum` + `estimate_defocus` on one.
- **Deep dives in this repository.** [`../../CTFEstimation/`](../../CTFEstimation/) (CTF estimation in depth),
  [`../../LowSNRNoiseProcessing/`](../../LowSNRNoiseProcessing/) (low-SNR processing, class averaging, denoising) and
  [`../../MotionCorrection/`](../../MotionCorrection/) (movie alignment and dose weighting).

## 📝 Lab note

```text
Module 06 — date:
Input: code-generated β-galactosidase-sized map (3.54 Å/px)
Output: projections, CTF curves, a simulated micrograph, Thon rings + defocus estimate, corrected images, class averages
Method: weak-phase/projection approximation; CTF (Δf>0 underfocus, CTF = −[√(1−A²)sinχ + A cosχ]E); Poisson counting;
        phase flipping; (multi-image) Wiener filtering; binning
Parameter I changed and what happened:
A failure mode / surprise (contrast inversion; CTF zeros; low-dose fit failure):
Paper link: Xu-lab simulator settings (Xu 2011/2012, Pei 2016, SHREC'20); CryoMAE (2025); TSM denoising (2026); Shi 2020
```

## 📚 References

1. Thon F. Zur Defokussierungsabhängigkeit des Phasenkontrastes bei der elektronenmikroskopischen Abbildung. *Z Naturforsch A* 21:476–478 (1966).
2. Erickson HP, Klug A. Measurement and compensation of defocusing and aberrations by Fourier processing of electron micrographs. *Phil Trans R Soc Lond B* 261:105–118 (1971). doi:10.1098/rstb.1971.0040
3. Rohou A, Grigorieff N. CTFFIND4: fast and accurate defocus estimation from electron micrographs. *J Struct Biol* 192:216–221 (2015). doi:10.1016/j.jsb.2015.08.008
4. Grant T, Grigorieff N. Measuring the optimal exposure for single particle cryo-EM using a 2.6 Å reconstruction of rotavirus VP6. *eLife* 4:e06980 (2015). doi:10.7554/eLife.06980
5. Dubochet J *et al.* Cryo-electron microscopy of vitrified specimens. *Q Rev Biophys* 21:129–228 (1988). doi:10.1017/S0033583500004297
6. Xu M, Beck M, Alber F. Template-free detection of macromolecular complexes in cryo electron tomograms. *Bioinformatics* 27:i69–i76 (2011). doi:10.1093/bioinformatics/btr207
7. Xu M, Beck M, Alber F. High-throughput subtomogram alignment and classification by Fourier space constrained fast volumetric matching. *J Struct Biol* 178:152–164 (2012). doi:10.1016/j.jsb.2012.02.014
8. Pei L, Xu M, Frazier Z, Alber F. Simulating cryo electron tomograms of crowded cell cytoplasm for assessment of automated particle picking. *BMC Bioinformatics* 17:405 (2016). doi:10.1186/s12859-016-1283-3
9. Gubins I *et al.* SHREC 2020: Classification in cryo-electron tomograms. *Computers & Graphics* 91:279–289 (2020). doi:10.1016/j.cag.2020.07.010
10. Xu C, Zhan X, Xu M. CryoMAE: Few-shot cryo-EM particle picking with masked autoencoders. *WACV* 2025, pp. 3876–3885. doi:10.1109/WACV61041.2025.00381
11. Score-Based Matching with Target Guidance for Cryo-EM Denoising. ECCV 2026 (authors not verified; code: AITom `aitom/filter/denoise/tsm/`).
12. Shi J, Zeng X, Jiang R, Jiang T, Xu M. A simulated annealing approach for resolution guided homogeneous cryo-electron microscopy image selection. *Quant Biol* 8:51–63 (2020). doi:10.1007/s40484-019-0191-8

## ➡️ Next

**[07 · Tomography: tilt series, back-projection and the missing wedge](07_tomography_missing_wedge.ipynb)** — tilt the
specimen, reconstruct it in 3D, and meet the wedge of information that is lost forever. Course overview:
[`README.md`](../README.md) · terms: [`GLOSSARY.md`](../GLOSSARY.md).